### **Step 1: Have the file "station_addresses_with_latlong_corrected.csv" in the same directory as the merged_dataset**

### **Step 2: MANUALLY create a file to track the progress, for example "progress_tracker.csv"**
**In this file, create the header for 3 columns: index,IncidentNumber,Distance**

Create a new line under the header line - IMPORTANT!!!

This is where the data is appended

### **Step 3: Partition of the merged_dataset based on its index (to be used in the code cell "The actual computation")** 

Laura: start_index = 0, end_index = 150000

Khoi: start_index = 150000, end_index = 300000

Yu part 1: start_index = 300000, end_index = 450000

Yu part 2: start_index = 450000, end_index = 608941

**Import necessary packages**

In [6]:
import pandas as pd
import requests
import time

**Reading the merged dataset, making sure the merged dataset has shape (608941, 25)**

In [2]:
df = pd.read_csv("merged_dataset.csv")
df.shape

(608941, 25)

**Reading the list of 102 stations**

In [3]:
stations = pd.read_csv("station_addresses_with_latlong_corrected.csv")
stations.shape

(102, 5)

**Capitalize station name, as the station name in the merged dataset is already capitalized**

In [4]:
stations["Station Name"] = stations["Station Name"].str.upper()

**Ensure data format, to prevent very small values (close to 0) being formatted in the scientifc notation**

In [5]:
df.Latitude = df.Latitude.map(lambda x: f"{x:.10f}")
df.Longitude = df.Longitude.map(lambda x: f"{x:.10f}")

**The function to get distance from OSM API**

In [9]:
def get_road_distance(lat1, lon1, lat2, lon2):
        
    url = f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {
        "overview": "false"  # no full route geometry needed
    }
    print(url)
    response = requests.get(url, params=params)
    data = response.json()

    distance_meters = -1 # If the distance cannot be computed, the distance will be -1
    duration_seconds = -1

    if data["code"] != "Ok":
        print("Error in API response:", data)  
    else:
        route = data["routes"][0]
        distance_meters = route["distance"]
        duration_seconds = route["duration"]

    return distance_meters, duration_seconds

**The dictionary storing station name as key and [lat, long] as value**

In [14]:
dict_station_latlong = {}

for i  in range(0, len(stations)):
    dict_station_latlong[stations["Station Name"][i]] = [stations["lat"][i].item(), stations["long"][i].item()]  # The order is [Lat, Long]

**The actual computation**

In [29]:
output_file_name = "progress_tracker.csv"

start_index = 
end_index = 

for i in range(start_index, end_index):
    incident_lat = df.Latitude[i]
    incident_long = df.Longitude[i]
    incident_number = df.IncidentNumber[i]

    station_lat  = dict_station_latlong[df.DeployedFromStation_Name[i]][0]
    station_long = dict_station_latlong[df.DeployedFromStation_Name[i]][1]

    distance, duration = get_road_distance(station_lat, station_long, incident_lat, incident_long)
    
    df.loc[i, "distance_fire_to_station"] = distance
    print(f"Iteration {i}, incidentNUmber is {incident_number} incident (lat-lon) is {incident_lat}, {incident_long} and station is {station_lat}, {station_long} distance is {distance} meter ")

    with open(output_file_name, "a") as file:
        file.write(f"{i},{incident_number},{distance}\n")

    time.sleep(0.005)

https://router.project-osrm.org/route/v1/driving/-0.4138853,51.4606344;-0.4135121214,51.4456983723
Iteration 150000, incidentNUmber is 093025-25062022 incident (lat-lon) is 51.4456983723, -0.4135121214 and station is 51.4606344, -0.4138853 distance is 2616.4 meter 
https://router.project-osrm.org/route/v1/driving/-0.078083,51.474087;-0.1028529381,51.4761537074
Iteration 150001, incidentNUmber is 093027-25062022 incident (lat-lon) is 51.4761537074, -0.1028529381 and station is 51.474087, -0.078083 distance is 2255.8 meter 
https://router.project-osrm.org/route/v1/driving/-0.3040197,51.5516966;-0.3075164135,51.5459465095
Iteration 150002, incidentNUmber is 093029-25062022 incident (lat-lon) is 51.5459465095, -0.3075164135 and station is 51.5516966, -0.3040197 distance is 1614.5 meter 
https://router.project-osrm.org/route/v1/driving/0.05732,51.523469;0.1603076599,51.5353146540
Iteration 150003, incidentNUmber is 093030-25062022 incident (lat-lon) is 51.5353146540, 0.1603076599 and statio

KeyboardInterrupt: 

### **Step 4: If the execution of the code cell stops for any reason, open the file "progress_tracker.csv" and look for the last index (last row) in the file. For example, last index in the file is 239385, then continue with the next index (239386) as the start_index in the for loop in the code cell "The actual computation". The end_index must not be changed**

### **Remark: While the distance computation is running, you can still open the file "progress_tracker.csv" (if needed). However, open it with Notepad or read the file's content by python. Opening this file with MS Excel will stop the distance computation, I don't know why**

### **Step 5: When you are done with the distance computation, simply send me your "progress_tracker.csv". I will merge all the results and create the merged_dataset with distance column filled**